# Data Query Workbook

Test queries that load futures and weather data from the SQLite warehouse.
Queries are defined in `eda/queries.py` for reuse outside this notebook.

In [1]:
import sys
from pathlib import Path

# Allow imports from project root
sys.path.insert(0, str(Path.cwd().parent))

from eda.queries import run, ALL_QUERIES

print("Available queries:", list(ALL_QUERIES.keys()))

Available queries: ['futures_all', 'futures_by_ticker', 'futures_latest', 'weather_all', 'weather_by_state', 'weather_latest', 'futures_weather_joined']


## Futures: coverage check

In [2]:
run("futures_latest")

,ticker,latest_date,row_count
0,ZC=F,2026-03-11,4068
1,ZS=F,2026-03-11,4070
2,ZW=F,2026-03-11,4070


## Futures: load single ticker

In [3]:
corn = run("futures_by_ticker", {"ticker": "ZC=F"})
print(f"{len(corn)} rows, {corn['date'].min().date()} to {corn['date'].max().date()}")
corn.head()

4068 rows, 2010-01-04 to 2026-03-11


,date,open,high,low,close,volume
0,2010-01-04,415.75,426.25,413.25,418.50,140553
1,2010-01-05,417.25,420.00,415.25,418.75,80967
2,2010-01-06,418.00,422.00,416.00,421.75,96453
3,2010-01-07,421.00,424.00,414.50,417.50,137395
4,2010-01-08,417.50,423.75,415.00,423.00,137395


## Weather: coverage check

In [4]:
run("weather_latest")

,state,latest_date,row_count
0,Illinois,2026-03-10,5913
1,Iowa,2026-03-10,5913
2,Nebraska,2026-03-10,5913


## Weather: load single state

In [5]:
iowa = run("weather_by_state", {"state": "Iowa"})
print(f"{len(iowa)} rows, {iowa['date'].min().date()} to {iowa['date'].max().date()}")
iowa.head()

5913 rows, 2010-01-01 to 2026-03-10


,date,temp_max_f,temp_min_f,precip_in
0,2010-01-01,3.92,-7.60,0.0
1,2010-01-02,-7.42,-18.40,0.0
2,2010-01-03,1.04,-9.94,0.0
3,2010-01-04,-0.94,-12.82,0.0
4,2010-01-05,0.68,-11.74,0.0


## Joined: futures + weather

In [6]:
joined = run("futures_weather_joined", {"ticker": "ZC=F"})
print(f"{len(joined)} rows, {joined['date'].min().date()} to {joined['date'].max().date()}")
joined.head(10)

12201 rows, 2010-01-04 to 2026-03-10


,date,close,state,temp_max_f,temp_min_f,precip_in
0,2010-01-04,418.50,Illinois,13.28,-0.04,0.000000
1,2010-01-04,418.50,Iowa,-0.94,-12.82,0.000000
2,2010-01-04,418.50,Nebraska,4.10,-8.68,0.000000
3,2010-01-05,418.75,Illinois,14.72,-0.94,0.000000
4,2010-01-05,418.75,Iowa,0.68,-11.74,0.000000
5,2010-01-05,418.75,Nebraska,7.34,-6.16,0.000000
6,2010-01-06,421.75,Illinois,22.82,4.10,0.051181
7,2010-01-06,421.75,Iowa,10.22,-10.84,0.137795
8,2010-01-06,421.75,Nebraska,18.86,6.08,0.137795
9,2010-01-07,417.50,Illinois,19.94,11.84,0.240157


In [7]:
## All futures (multi-ticker)

In [8]:
all_futures = run("futures_all")
print(f"{len(all_futures)} total rows")
all_futures.groupby("ticker").agg(
    rows=("date", "count"),
    first=("date", "min"),
    last=("date", "max"),
)

12208 total rows


,rows,first,last
ticker,,,
ZC=F,4068,2010-01-04,2026-03-11
ZS=F,4070,2010-01-04,2026-03-11
ZW=F,4070,2010-01-04,2026-03-11


## Feature Store Verification

Quick sample from every Parquet table via DuckDB.

In [9]:
from features.query import query, list_tickers, list_features, get_ticker_features, get_unlinked_features

# All available tickers
list_tickers()

[{'symbol': 'ZC=F',
  'name': 'corn',
  'source': 'yahoo_finance',
  'asset_class': 'agricultural_futures',
  'description': 'CBOT Corn Futures continuous contract'},
 {'symbol': 'ZS=F',
  'name': 'soybeans',
  'source': 'yahoo_finance',
  'asset_class': 'agricultural_futures',
  'description': 'CBOT Soybean Futures continuous contract'},
 {'symbol': 'ZW=F',
  'name': 'wheat',
  'source': 'yahoo_finance',
  'asset_class': 'agricultural_futures',
  'description': 'CBOT Wheat Futures continuous contract'}]

In [ ]:
# Momentum: corn sample
query("SELECT * FROM 'features/momentum/corn.parquet' WHERE sma_20 IS NOT NULL LIMIT 3")

In [ ]:
# Mean reversion: soybeans sample
query("SELECT * FROM 'features/mean_reversion/soybeans.parquet' WHERE zscore_20 IS NOT NULL LIMIT 3")

In [12]:
# Weather: iowa sample
query("SELECT * FROM 'features/weather/iowa.parquet' WHERE precip_7d IS NOT NULL LIMIT 3")

,date,precip_7d,precip_30d,temp_max_7d,temp_min_7d,temp_range_7d,precip_anomaly_30d
0,2010-01-07,0.224409,NaN,2.840000,-9.605714,12.445714,NaN
1,2010-01-08,0.224409,NaN,2.762857,-9.734286,12.497143,NaN
2,2010-01-09,0.224409,NaN,3.508571,-9.220000,12.728571,NaN


In [13]:
# Ticker-feature map: what features does corn have?
get_ticker_features("corn")

{'momentum': ['sma_20',
  'sma_50',
  'ema_12',
  'ema_26',
  'macd',
  'macd_signal',
  'rsi_14'],
 'mean_reversion': ['bollinger_upper',
  'bollinger_lower',
  'zscore_20',
  'zscore_50',
  'pct_rank_20']}

In [14]:
# Unlinked features (weather -- no ticker)
get_unlinked_features()

{'weather': {'iowa': ['precip_7d',
   'precip_30d',
   'temp_max_7d',
   'temp_min_7d',
   'temp_range_7d',
   'precip_anomaly_30d'],
  'illinois': ['precip_7d',
   'precip_30d',
   'temp_max_7d',
   'temp_min_7d',
   'temp_range_7d',
   'precip_anomaly_30d'],
  'nebraska': ['precip_7d',
   'precip_30d',
   'temp_max_7d',
   'temp_min_7d',
   'temp_range_7d',
   'precip_anomaly_30d']}}

In [ ]:
# Cross-join: corn momentum + iowa weather
query("""
    SELECT m.date, m.sma_20, m.rsi_14, w.precip_30d
    FROM 'features/momentum/corn.parquet' m
    JOIN 'features/weather/iowa.parquet' w ON m.date = w.date
    WHERE m.sma_20 IS NOT NULL
    LIMIT 3
""")